# Práctica 1 — Redes Neuronales Convolucionales (CNN)
**Materia:** Deep Learning  
**Maestría en Ciencias en Inteligencia Artificial — UAQ**

## Objetivo
Diseñar, entrenar y evaluar una CNN para clasificación de imágenes sobre el dataset MNIST, analizando el efecto del número de filtros y la profundidad de la red.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Carga y preprocesamiento de datos

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

X_train = X_train[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test, 10)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for ax, img, label in zip(axes, X_train[:8], y_train[:8]):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label)); ax.axis('off')
plt.suptitle('Ejemplos del dataset MNIST', y=1.05)
plt.savefig('mnist_samples.png', dpi=100)
plt.show()

## 2. Definición de la arquitectura CNN

In [ ]:
def build_cnn(num_filters=32, num_layers=2):
    model = models.Sequential(name=f'CNN_{num_layers}L_{num_filters}F')
    model.add(layers.Input(shape=(28, 28, 1)))
    for i in range(num_layers):
        model.add(layers.Conv2D(num_filters * (2**i), (3,3), activation='relu', padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2,2)))
        model.add(layers.Dropout(0.25))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(10, activation='softmax'))
    return model

modelo = build_cnn(num_filters=32, num_layers=2)
modelo.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
modelo.summary()

## 3. Entrenamiento

In [ ]:
historia = modelo.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## 4. Evaluación y visualización

In [ ]:
loss, acc = modelo.evaluate(X_test, y_test_cat, verbose=0)
print(f"Precisión en prueba: {acc*100:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(historia.history['accuracy'], label='Entrenamiento')
ax1.plot(historia.history['val_accuracy'], label='Validación')
ax1.set_title('Precisión por época'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(historia.history['loss'], label='Entrenamiento')
ax2.plot(historia.history['val_loss'], label='Validación')
ax2.set_title('Pérdida por época'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cnn_training.png', dpi=120)
plt.show()

## Conclusiones
- La arquitectura CNN con dos bloques convolucionales alcanza >99% de precisión en MNIST.
- BatchNormalization acelera la convergencia y Dropout reduce el sobreajuste.
- La adición de más capas mejora la capacidad del modelo pero aumenta el costo computacional.